# FinRisk Radar — Notebook 01: Exploratory Data Analysis

This notebook explores the financial dataset and validates the feature engineering pipeline.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
pd.set_option('display.float_format', '{:.3f}'.format)
print('Imports OK')

## Step 1: Load and inspect raw data

In [ ]:
# First run data ingestion
# from data.ingestion import ingest_multiple_tickers, SAMPLE_TICKERS
# df_raw = ingest_multiple_tickers(SAMPLE_TICKERS)

# Load if already ingested
raw_path = Path('../data/raw/all_financials.csv')
if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
    print(f'Shape: {df_raw.shape}')
    print(f'Tickers: {df_raw["ticker"].unique()}')
    df_raw.head()

## Step 2: Feature engineering

In [ ]:
from ml.feature_engineering import build_feature_dataset, FEATURE_COLS

df = build_feature_dataset()
print(f'Feature dataset shape: {df.shape}')
print(f'\nDistress label distribution:')
print(df['distressed'].value_counts())
print(f'\nAltman Zone distribution:')
print(df['altman_zone'].value_counts())

## Step 3: Feature correlation heatmap

In [ ]:
available_feats = [c for c in FEATURE_COLS if c in df.columns]
corr = df[available_feats + ['distressed']].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Altman Z-Score distribution by distress label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Z-Score distribution
for label, grp in df.groupby('distressed'):
    axes[0].hist(grp['altman_z'].dropna().clip(-5, 10),
                 bins=30, alpha=0.6,
                 label='Distressed' if label else 'Safe',
                 color='#ef4444' if label else '#10b981')
axes[0].axvline(1.81, color='orange', linestyle='--', label='Z=1.81 (distress threshold)')
axes[0].axvline(2.99, color='cyan',   linestyle='--', label='Z=2.99 (safe threshold)')
axes[0].set_title('Altman Z-Score Distribution')
axes[0].legend()

# Altman zone pie
zone_counts = df['altman_zone'].value_counts()
axes[1].pie(zone_counts, labels=zone_counts.index,
            colors=['#10b981', '#f59e0b', '#ef4444'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Altman Zone Distribution')

plt.tight_layout()
plt.savefig('../data/processed/eda_distributions.png', dpi=150)
plt.show()

## Step 5: Top risk companies in our sample

In [ ]:
top_risk = df.nsmallest(10, 'altman_z')[['ticker','date','altman_z','altman_zone','debt_to_equity','current_ratio','distressed']]
print('Top 10 Highest Risk Companies (lowest Altman Z):')
top_risk